# US Rates & SOFR Pricing Engine — End-to-End Analysis

**Bhavesh Anchalia**

This notebook walks through the full analysis pipeline:
1. Data fetch (FRED + CME)
2. SOFR curve bootstrapping
3. Instrument pricing validation
4. Nelson-Siegel Treasury curve fitting
5. Taylor Rule / Fed policy analysis
6. Signal construction
7. Walk-forward backtest
8. Paper-quality charts

See `BRAIN.md` for research notes and paper draft.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# FRED API key (set environment variable or paste here for local dev)
FRED_API_KEY = os.environ.get('FRED_API_KEY', 'YOUR_KEY_HERE')

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
print('Setup complete')

## 1. Data Fetch

In [ ]:
from data.fetch_sofr_data import fetch_fred, add_derived_series
from data.fetch_treasury_data import fetch_treasury_yields, compute_spreads

# Fetch all FRED data (SOFR, Fed Funds, macro, Treasury yields)
macro_df   = fetch_fred(FRED_API_KEY, start_date='2018-01-01')
macro_df   = add_derived_series(macro_df)

tsy_df     = fetch_treasury_yields(FRED_API_KEY, start_date='2015-01-01')
tsy_df     = compute_spreads(tsy_df)

print(f'Macro data: {macro_df.shape[0]} rows × {macro_df.shape[1]} cols')
print(f'Treasury data: {tsy_df.shape[0]} rows × {tsy_df.shape[1]} cols')
macro_df[['sofr', 'fed_funds', 'iorb']].tail(10)

## 2. SOFR Curve Bootstrap

We build a SOFR OIS forward curve from:
- Overnight SOFR fixing
- CME SR3 futures (with Hull-White convexity adjustment)
- SOFR OIS swap quotes for tenors beyond 2Y

In [ ]:
from sofr_engine import SOFRCurveBootstrapper
from sofr_engine.convexity import hull_white_convexity_adjustment

# Example: build curve as of 2024-06-05 (near peak rate environment)
ref_date   = date(2024, 6, 5)
sofr_on    = 0.0530  # 5.30% overnight SOFR

# Synthetic SR3 futures strip (replace with actual CME data)
import pandas as pd
futures_strip = pd.DataFrame([
    {'expiry': date(2024, 9, 18),  'accrual_end': date(2024, 12, 18), 'implied_rate': 0.0520},
    {'expiry': date(2024, 12, 18), 'accrual_end': date(2025, 3, 19),  'implied_rate': 0.0492},
    {'expiry': date(2025, 3, 19),  'accrual_end': date(2025, 6, 18),  'implied_rate': 0.0462},
    {'expiry': date(2025, 6, 18),  'accrual_end': date(2025, 9, 17),  'implied_rate': 0.0438},
    {'expiry': date(2025, 9, 17),  'accrual_end': date(2025, 12, 17), 'implied_rate': 0.0418},
    {'expiry': date(2025, 12, 17), 'accrual_end': date(2026, 3, 18),  'implied_rate': 0.0400},
    {'expiry': date(2026, 3, 18),  'accrual_end': date(2026, 6, 17),  'implied_rate': 0.0388},
    {'expiry': date(2026, 6, 17),  'accrual_end': date(2026, 9, 16),  'implied_rate': 0.0378},
])

# OIS quotes for longer tenors
ois_quotes = [
    (2.0,  0.0448),
    (3.0,  0.0435),
    (5.0,  0.0418),
    (7.0,  0.0410),
    (10.0, 0.0405),
    (15.0, 0.0402),
    (20.0, 0.0400),
    (30.0, 0.0398),
]

curve = SOFRCurveBootstrapper.from_market_data(
    ref_date, sofr_on, futures_strip, ois_quotes, sigma=0.010
)
print(curve)
print('\nZero Curve:')
curve.zero_curve()

In [ ]:
from visualisation.charts import sofr_forward_curve
sofr_forward_curve(curve, label='2024-06-05 (near peak rates)')

## 3. Instrument Pricing Validation

Key test: a **par swap at inception must have PV = 0**.

In [ ]:
from sofr_engine import SOFRSwap, ForwardRateAgreement

# 5Y SOFR payer swap at par
par_5y = curve.par_ois_rate(5.0)
swap_5y = SOFRSwap(
    effective_date=date(2024, 6, 7),
    maturity_date=date(2029, 6, 7),
    fixed_rate=par_5y,
    notional=10_000_000,
    pay_fixed=True,
)

summary = swap_5y.summary(curve)
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k:<25}: {v:>12.4f}')
    else:
        print(f'  {k:<25}: {v}')

assert abs(summary['net_pv']) < 10, 'Par swap PV should be ~$0'
print('\n✓ Par swap validation passed')

In [ ]:
# Convexity adjustment table
from sofr_engine.convexity import hull_white_convexity_adjustment

print('Hull-White Convexity Adjustments (σ=1%)')
print(f'{"Contract":<15} {"T1":>6} {"T2":>6} {"CA (bps)":>10} {"Futures Rate":>14} {"Forward Rate":>14}')
print('-' * 70)

for _, row in futures_strip.iterrows():
    t1 = (row['expiry']      - ref_date).days / 365.25
    t2 = (row['accrual_end'] - ref_date).days / 365.25
    ca = hull_white_convexity_adjustment(t1, t2, sigma=0.010)
    f_fut = row['implied_rate']
    f_fwd = f_fut - ca
    print(f"{str(row['expiry']):<15} {t1:>6.2f} {t2:>6.2f} {ca*10000:>10.3f} {f_fut*100:>14.4f}% {f_fwd*100:>14.4f}%")

## 4. Nelson-Siegel Treasury Curve Fitting

In [ ]:
from models.nelson_siegel import fit_nelson_siegel, ns_yield, rolling_ns_factors, classify_curve_regime
from data.fetch_treasury_data import MATURITY_LABELS, STANDARD_MATURITIES

# Columns from tsy_df corresponding to standard maturities
maturity_cols  = ['tsy_1m','tsy_3m','tsy_6m','tsy_1y','tsy_2y','tsy_3y',
                  'tsy_5y','tsy_7y','tsy_10y','tsy_20y','tsy_30y']
maturities_yrs = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]

# Fit NS to most recent curve
available_cols = [c for c in maturity_cols if c in tsy_df.columns]
latest = tsy_df[available_cols].dropna(how='all').iloc[-1]
mat_arr = [m for c, m in zip(maturity_cols, maturities_yrs) if c in available_cols]
yld_arr = latest.values.astype(float)
valid   = ~np.isnan(yld_arr)

params = fit_nelson_siegel(np.array(mat_arr)[valid], yld_arr[valid])
print(f'Latest date: {tsy_df.index[-1].date()}')
print(f'NS fit: {params}')

t_fit = np.linspace(0.1, 30, 300)
y_fit = ns_yield(t_fit, params)

plt.figure(figsize=(10,4))
plt.scatter(np.array(mat_arr)[valid], yld_arr[valid], s=60, zorder=5, label='Observed CMT')
plt.plot(t_fit, y_fit, linewidth=2, label=f'NS fit (β₀={params.beta0:.2f}%, β₁={params.beta1:.2f}%, β₂={params.beta2:.2f}%)')
plt.xlabel('Maturity (years)')
plt.ylabel('Yield (%)')
plt.title('US Treasury Yield Curve — Nelson-Siegel Fit')
plt.legend()
plt.show()

In [ ]:
# Rolling NS factors over time
# Sample every week for speed; use all business days for final paper
tsy_weekly = tsy_df[available_cols].resample('W').last()

ns_factors = rolling_ns_factors(
    tsy_weekly,
    maturity_cols=available_cols,
    maturities=mat_arr,
)
ns_factors['regime'] = classify_curve_regime(ns_factors)

print(f'NS factors computed for {len(ns_factors)} weeks')
print('\nRegime distribution:')
print(ns_factors['regime'].value_counts())

from visualisation.charts import ns_factor_history
ns_factor_history(ns_factors)

## 5. Taylor Rule Analysis

In [ ]:
from models.taylor_rule import compute_taylor_rule, TaylorRuleConfig, taylor_policy_gap

# Use core PCE YoY and unemployment from macro_df
taylor_df = compute_taylor_rule(
    inflation=macro_df['core_pce_yoy'].dropna(),
    unemployment=macro_df['unemployment'].dropna(),
)

from visualisation.charts import taylor_rule_vs_actual
taylor_rule_vs_actual(
    actual_rate=macro_df['fed_funds'].dropna(),
    taylor_df=taylor_df,
)

## 6. FOMC Probabilities

In [ ]:
from models.fomc_probability import fomc_prob_summary, next_fomc, fedwatch_probabilities

current_rate  = 0.0530  # Current Fed Funds target (5.30%)
implied_rates = {
    'Jun-25': 0.0505,
    'Jul-25': 0.0498,
    'Sep-25': 0.0488,
    'Nov-25': 0.0475,
}

print('Market-Implied FOMC Probabilities:')
print(f'{"Meeting":<12} {"Cut":>8} {"Hold":>8} {"Hike":>8} {"Implied Rate":>14}')
print('-' * 55)
for meeting, impl_rate in implied_rates.items():
    p = fomc_prob_summary(impl_rate, current_rate)
    print(f'{meeting:<12} {p["p_cut"]:>8.1%} {p["p_hold"]:>8.1%} {p["p_hike"]:>8.1%} {impl_rate*100:>13.2f}%')

## 7. Signal Construction

In [ ]:
from models.macro_signals import composite_signal

# Merge taylor_rate into macro_df
combined = macro_df.copy()
combined['taylor_rate'] = taylor_df['taylor_rate'].reindex(combined.index, method='ffill')

# Also need slope
for c in ['tsy_2y', 'tsy_10y', 'slope_2s10s']:
    if c in tsy_df.columns:
        combined[c] = tsy_df[c].reindex(combined.index, method='ffill')

signal_df = composite_signal(combined)

print('Signal distribution:')
print(signal_df['position'].value_counts())
print(f'\nSignal agreement (avg): {signal_df["signal_agreement"].mean():.2f}')

signal_df[['composite_score', 'position']].resample('M').last().tail(24)

## 8. Walk-Forward Backtest

In [ ]:
from backtesting.signal_backtest import WalkForwardBacktest, make_signal_fn
from backtesting.performance import full_metrics, print_tearsheet

# Build backtest dataset: needs tsy_10y for return computation
backtest_data = combined.copy()

bt = WalkForwardBacktest(
    train_window=252,
    test_window=63,
    txn_cost_bps=0.5,
    duration=8.0,
)

signal_fn = make_signal_fn()

result = bt.run(
    data=backtest_data,
    signal_fn=signal_fn,
    yield_col='tsy_10y',
)

print(f'Backtest period: {result.index[0].date()} to {result.index[-1].date()}')
print(f'Total days: {len(result)}')

metrics = full_metrics(result)
print_tearsheet(metrics, 'US Rates Signal Strategy — Walk-Forward')

In [ ]:
from visualisation.charts import backtest_pnl
backtest_pnl(result)

## 9. Key Results Summary

Numbers below reflect live FRED data from the paper (2018–2025 sample).
The SOFR curve metrics use the static Jun-2024 snapshot built above.

In [ ]:
print('=== KEY RESULTS SUMMARY ===')
# SOFR curve (from static snapshot above — no live FRED needed)
print(f'SOFR curve pillars (Jun-2024 snapshot): {len(curve._times)}')
print(f'5Y par OIS rate: {curve.par_ois_rate(5.0)*100:.4f}%')
print(f'10Y par OIS rate: {curve.par_ois_rate(10.0)*100:.4f}%')

# Known results from walk-forward backtest (from paper draft)
print()
print('=== Walk-Forward Backtest (2019-2025, 63d test windows) ===')
print(f'OOS Sharpe ratio:          0.282  (35bp stop-loss)')
print(f'Hit rate:                 65.7%')
print(f'Win/loss ratio:            1.21')
print(f'Max drawdown:           -142bps')
print()
print('=== Taylor Rule (Standard, Jun-2026 snapshot) ===')
print(f'Core PCE YoY:              2.11%')
print(f'Unemployment rate:         4.30%')
print(f'Taylor recommended rate:   2.26%')
print(f'EFFR (actual):             4.33%')
print(f'Policy gap:              +138bps  (Fed 138bps above Taylor)')
print()
print('=== Nelson-Siegel Analysis ===')
print(f'Inversion episode:   105 weeks (Nov-2022 to Dec-2024)')
print(f'Peak inversion:      beta1 = +1.74% (Mar-2023)')
print(f'Current regime:      normal_flat (beta1 ~ -0.2%)')
print()
print('=== SOFR-T-Bill Basis ===')
print(f'Mean spread: 0.09bps  (effectively zero post-LIBOR transition)')
